# Phase 3 — Exploratory Data Analysis
## `ML_ready_PLGA.csv` — structure of EE% and its relationship to molecular / PLGA descriptors

**Input:** `data/processed/ML_ready_PLGA.csv` (produced by `notebooks/02_preprocessing.ipynb`).
**Output:** three publication-quality figures at **300 DPI** in `results/figures/`, plus supporting tables in `results/tables/`.

**No model is trained in this notebook.** This is descriptive analysis only.

### Required figures
| File | Content |
|---|---|
| `Fig1_correlation_heatmap_EE.png` | Correlation heatmap of all continuous molecular + PLGA descriptors against EE% |
| `Fig2_logP_vs_EE_by_LAGA.png` | `mol_logP` (lipophilicity) vs EE%, colour-coded by PLGA LA/GA ratio |
| `Fig3_EE_distribution.png` | Distribution of EE% |

### Interpretation guardrail
Every relationship below is an **observational correlation** pooled across 59 independent published studies. Correlation is **not** causation: apparent descriptor–EE trends are confounded with study protocol, polymer grade, and which drugs each lab happened to study. Nothing here is a clinical or mechanistic claim.

In [1]:
# --- Setup ---
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
})

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise RuntimeError("Could not locate project root (folder containing data/raw).")

ROOT = find_root(Path.cwd())
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "results" / "figures"
TAB  = ROOT / "results" / "tables"
for d in (FIG, TAB): d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Figures ->", FIG)

Project root: C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization
Figures -> C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization\results\figures


## 1. Load the ML-ready dataset and restore interpretable units
`ML_ready_PLGA.csv` stores continuous predictors as **z-scores** (StandardScaler) and `pH` / `LA/GA` as **one-hot** columns. Standardisation is a linear transform, so it leaves Pearson correlations unchanged — but it makes axes unreadable. We therefore:

1. load `ML_ready_PLGA.csv` (the ML-ready file, as the single source of truth),
2. invert the z-scores using the saved `scaler_params.csv` to recover original units for plotting,
3. reconstruct the numeric `LA/GA` ratio and the `pH` code from their one-hot columns,
4. **assert** the reconstruction matches the independently saved unscaled companion, to prove no distortion was introduced.

`EE` itself was never scaled — it is stored in raw % units.

In [2]:
ml   = pd.read_csv(PROC / "ML_ready_PLGA.csv")
meta = json.loads((PROC / "preprocessing_meta.json").read_text(encoding="utf-8"))
sp   = pd.read_csv(PROC / "scaler_params.csv").set_index("feature")

TARGET     = meta["target"]
CONTINUOUS = meta["continuous_features"]
ONEHOT     = meta["onehot_output_columns"]
GROUP_KEY  = meta["group_key_for_cv"]

print(f"ML_ready_PLGA.csv: {ml.shape[0]} formulations x {ml.shape[1]} columns")
print(f"target = {TARGET} | {len(CONTINUOUS)} scaled continuous + {len(ONEHOT)} one-hot predictors")
print(f"drugs = {ml['drug_key'].nunique()} | drug_groups (CV key '{GROUP_KEY}') = {ml[GROUP_KEY].nunique()}")
print(f"studies (reference) = {ml['reference'].nunique()}")
print(f"leakage columns absent from file: {[c for c in meta['leakage_dropped'] if c not in ml.columns]}")
assert not any(c in ml.columns for c in meta["leakage_dropped"]), "Leakage column present — abort."
assert ml[TARGET].notna().all() and ml.isna().sum().sum() == 0

# --- invert standardisation to original units (plot-only) ---
orig = (ml[CONTINUOUS]
        .mul(sp.loc[CONTINUOUS, "scale"], axis=1)
        .add(sp.loc[CONTINUOUS, "mean"],  axis=1))

# --- reconstruct LA/GA ratio and pH code from one-hot columns ---
laga_cols = [c for c in ONEHOT if c.startswith("LAGA_")]
ph_cols   = [c for c in ONEHOT if c.startswith("pH_")]
assert np.allclose(ml[laga_cols].sum(axis=1), 1), "LA/GA one-hot rows must sum to 1."
assert np.allclose(ml[ph_cols].sum(axis=1),   1), "pH one-hot rows must sum to 1."
orig["LA_GA_ratio"] = ml[laga_cols].idxmax(axis=1).str.replace("LAGA_", "", regex=False).astype(float)
ph_code = ml[ph_cols].idxmax(axis=1).str.replace("pH_", "", regex=False)

eda = pd.concat([ml[["row_id", "small_molecule_name", "drug_key", GROUP_KEY,
                     "reference", "EE_is_zero", TARGET]], orig], axis=1)
eda["pH_code"] = ph_code.to_numpy()

# --- integrity: reconstruction must match the independently saved unscaled companion ---
unsc = pd.read_csv(PROC / "PLGA_clean_unscaled.csv")
for c in CONTINUOUS:
    assert np.allclose(eda[c].to_numpy(float), unsc[c].to_numpy(float), atol=1e-8), f"De-scaling mismatch: {c}"
assert np.allclose(eda["LA_GA_ratio"].to_numpy(float), unsc["LA/GA"].to_numpy(float)), "LA/GA reconstruction mismatch"
print("\nIntegrity PASS: de-scaled values and reconstructed LA/GA match the unscaled companion exactly.")
display(eda.head(4))

ML_ready_PLGA.csv: 430 formulations x 31 columns
target = EE | 13 scaled continuous + 9 one-hot predictors
drugs = 65 | drug_groups (CV key 'drug_group') = 63
studies (reference) = 59
leakage columns absent from file: ['LC', 'particle_size']

Integrity PASS: de-scaled values and reconstructed LA/GA match the unscaled companion exactly.


,row_id,small_molecule_name,drug_key,drug_group,reference,EE_is_zero,EE,mol_MW,mol_logP,mol_TPSA,mol_melting_point,mol_Hacceptors,mol_Hdonors,mol_heteroatoms,polymer_MW,drug/polymer,surfactant_concentration,aqueous/organic,surfactant_HLB,solvent_polarity_index,LA_GA_ratio,pH_code
0,0,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,False,25.18,392.41,2.08,100.59,267.5,6.0,3.0,9.0,14.0,1.0,1.5,4.0,18.0,5.1,1.0,0
1,1,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,False,44.83,392.41,2.08,100.59,267.5,6.0,3.0,9.0,14.0,0.5,1.5,4.0,18.0,5.1,1.0,0
2,2,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,False,51.83,392.41,2.08,100.59,267.5,6.0,3.0,9.0,14.0,0.2,1.5,4.0,18.0,5.1,1.0,0
3,3,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,False,86.60,392.41,2.08,100.59,267.5,6.0,3.0,9.0,14.0,0.1,1.5,4.0,18.0,5.1,1.0,0


## 2. EE% — descriptive statistics and the flagged total-failure formulation
Per the Phase-2 decision, the single **EE = 0** formulation was **retained** (total encapsulation failure is a genuine experimental outcome, not a data error). It is flagged here so its influence on any downstream fit is visible rather than silent.

In [3]:
ee = eda[TARGET]
ee_stats = pd.Series({
    "n": int(ee.size), "missing": int(ee.isna().sum()),
    "mean": ee.mean(), "std": ee.std(), "min": ee.min(),
    "q05": ee.quantile(.05), "q25": ee.quantile(.25), "median": ee.median(),
    "q75": ee.quantile(.75), "q95": ee.quantile(.95), "max": ee.max(),
    "IQR": ee.quantile(.75) - ee.quantile(.25),
    "skewness": float(stats.skew(ee)), "excess_kurtosis": float(stats.kurtosis(ee)),
    "shapiro_p": float(stats.shapiro(ee).pvalue),
    "n_EE_eq_0": int((ee == 0).sum()), "n_EE_below_20": int((ee < 20).sum()),
    "n_EE_above_90": int((ee > 90).sum()),
})
ee_stats.to_frame("value").to_csv(TAB / "eda_EE_summary.csv")
print(ee_stats.round(3).to_string())
print("\nEE% is left-skewed (skew < 0) and non-normal (Shapiro p < 0.05):")
print("most formulations succeed at high EE, with a thin low-EE tail. Report median/IQR alongside mean/SD,")
print("and prefer error metrics that are not dominated by the tail (MAE as well as RMSE) in Phase 4.")

print("\n--- FLAGGED: retained total-encapsulation-failure formulation (EE = 0) ---")
display(eda.loc[eda["EE_is_zero"], ["row_id", "small_molecule_name", "reference", TARGET,
                                    "mol_logP", "drug/polymer", "LA_GA_ratio", "surfactant_concentration"]])

n                  430.000
missing              0.000
mean                64.788
std                 23.509
min                  0.000
q05                  8.757
q25                 52.418
median              70.640
q75                 83.180
q95                 91.532
max                 98.900
IQR                 30.762
skewness            -0.997
excess_kurtosis      0.309
shapiro_p            0.000
n_EE_eq_0            1.000
n_EE_below_20       32.000
n_EE_above_90       36.000

EE% is left-skewed (skew < 0) and non-normal (Shapiro p < 0.05):
most formulations succeed at high EE, with a thin low-EE tail. Report median/IQR alongside mean/SD,
and prefer error metrics that are not dominated by the tail (MAE as well as RMSE) in Phase 4.

--- FLAGGED: retained total-encapsulation-failure formulation (EE = 0) ---


,row_id,small_molecule_name,reference,EE,mol_logP,drug/polymer,LA_GA_ratio,surfactant_concentration
369,372,pranoprofen,10.1002/jps.24101,0.0,2.97,0.125,3.0,1.5


## 3. Figure 1 — correlation of continuous molecular & PLGA descriptors with EE%
Left panel: full Pearson correlation matrix over the 13 continuous predictors, the numeric `LA/GA` ratio, and `EE`. Right panel: each descriptor's correlation with EE, sorted by magnitude, with Spearman ρ overlaid.

Why both coefficients: EE% is bounded and left-skewed (Section 2), so Pearson *r* (linear, outlier-sensitive) and Spearman ρ (monotonic, rank-based) can disagree — where they do, the relationship is non-linear or outlier-driven. `LA/GA` takes only 5 discrete values; it is included numerically for correlation purposes only (it is one-hot encoded for modelling).

In [4]:
HEAT_VARS = CONTINUOUS + ["LA_GA_ratio", TARGET]
PRETTY = {
    "mol_MW": "Mol. weight", "mol_logP": "logP", "mol_TPSA": "TPSA",
    "mol_melting_point": "Melting point", "mol_Hacceptors": "H-acceptors",
    "mol_Hdonors": "H-donors", "mol_heteroatoms": "Heteroatoms",
    "polymer_MW": "Polymer MW", "drug/polymer": "Drug/polymer",
    "surfactant_concentration": "Surfactant conc.", "aqueous/organic": "Aqueous/organic",
    "surfactant_HLB": "Surfactant HLB", "solvent_polarity_index": "Solvent polarity",
    "LA_GA_ratio": "LA/GA ratio", "EE": "EE (%)",
}
labels = [PRETTY[v] for v in HEAT_VARS]

C = eda[HEAT_VARS].corr(method="pearson")
S = eda[HEAT_VARS].corr(method="spearman")

# Correlation-with-EE table (both coefficients + p-values), sorted by |Pearson r|
rows = []
for v in CONTINUOUS + ["LA_GA_ratio"]:
    r, rp = stats.pearsonr(eda[v], eda[TARGET])
    rho, sp_p = stats.spearmanr(eda[v], eda[TARGET])
    rows.append({"descriptor": v, "pearson_r": r, "pearson_p": rp,
                 "spearman_rho": rho, "spearman_p": sp_p, "abs_pearson_r": abs(r)})
corr_ee = (pd.DataFrame(rows).sort_values("abs_pearson_r", ascending=False)
           .reset_index(drop=True))
corr_ee.drop(columns="abs_pearson_r").to_csv(TAB / "eda_correlations_with_EE.csv", index=False)
C.to_csv(TAB / "eda_correlation_matrix_pearson.csv")
display(corr_ee.drop(columns="abs_pearson_r").round(4))

fig = plt.figure(figsize=(15.8, 7.2))
gs  = fig.add_gridspec(1, 2, width_ratios=[1.32, 1.0], wspace=0.50)

# ---- (a) heatmap ----
axA = fig.add_subplot(gs[0, 0])
axA.grid(False)
M = C.to_numpy()
im = axA.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
n = len(HEAT_VARS)
axA.set_xticks(range(n)); axA.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
axA.set_yticks(range(n)); axA.set_yticklabels(labels, fontsize=8)
for i in range(n):
    for j in range(n):
        val = M[i, j]
        axA.text(j, i, f"{val:.2f}".replace("0.", ".").replace("-.", "−."),
                 ha="center", va="center", fontsize=6.4,
                 color="white" if abs(val) > 0.55 else "#222222")
# outline the EE row/column (the target)
k = HEAT_VARS.index(TARGET)
for (x, y, w, h) in [(-0.5, k - 0.5, n, 1), (k - 0.5, -0.5, 1, n)]:
    axA.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="black", lw=1.8))
axA.set_title("(a) Pearson correlation matrix\n(EE row/column outlined)", fontsize=10.5)
cb = fig.colorbar(im, ax=axA, fraction=0.046, pad=0.03)
cb.set_label("Pearson r", fontsize=9); cb.ax.tick_params(labelsize=8)

# ---- (b) correlation with EE, sorted ----
axB = fig.add_subplot(gs[0, 1])
d = corr_ee.iloc[::-1]                       # largest |r| at top
ypos = np.arange(len(d))
cols = ["#c0392b" if v > 0 else "#2471a3" for v in d["pearson_r"]]
axB.barh(ypos, d["pearson_r"], color=cols, edgecolor="black", linewidth=0.5, height=0.66, label="Pearson r")
axB.scatter(d["spearman_rho"], ypos, marker="D", s=26, facecolor="white",
            edgecolor="black", zorder=5, linewidth=0.9, label="Spearman ρ")
axB.set_yticks(ypos)
axB.set_yticklabels([PRETTY[v] for v in d["descriptor"]], fontsize=8.5)
axB.axvline(0, color="black", lw=1)
for thr in (-0.3, 0.3):
    axB.axvline(thr, color="grey", ls=":", lw=1)
axB.set_xlim(-0.6, 0.6)
axB.set_xlabel("Correlation with EE (%)")
axB.set_title("(b) Descriptor–EE association\n(dotted lines: |coef| = 0.3)", fontsize=10.5)
# significance stars on the Pearson bars
for yv, (r, p) in enumerate(zip(d["pearson_r"], d["pearson_p"])):
    star = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
    if star:
        axB.text(r + (0.02 if r > 0 else -0.02), yv, star, va="center",
                 ha="left" if r > 0 else "right", fontsize=8)
axB.legend(loc="lower right", fontsize=8, framealpha=0.95)

fig.suptitle("Figure 1. Continuous molecular and PLGA descriptors vs encapsulation efficiency "
             f"(n = {len(eda)} formulations, {eda[GROUP_KEY].nunique()} drug groups, "
             f"{eda['reference'].nunique()} studies)",
             fontsize=12, y=1.02)
fig.text(0.5, -0.045,
         "Observational correlations pooled across independent published studies; * p<0.05, ** p<0.01, *** p<0.001. "
         "Association is not causation. LC and particle_size are excluded as target-leaking / outcome variables.",
         ha="center", fontsize=8, style="italic", color="#444444")
out1 = FIG / "Fig1_correlation_heatmap_EE.png"
fig.savefig(out1, dpi=300); plt.close(fig)
print("Saved:", out1.relative_to(ROOT), "| 300 DPI")

,descriptor,pearson_r,pearson_p,spearman_rho,spearman_p
0,surfactant_concentration,0.3118,0.0000,0.2849,0.0000
1,surfactant_HLB,0.3013,0.0000,0.1985,0.0000
2,mol_logP,0.2272,0.0000,0.2346,0.0000
3,LA_GA_ratio,0.1907,0.0001,0.2441,0.0000
4,polymer_MW,0.1598,0.0009,0.2810,0.0000
5,mol_Hdonors,0.1309,0.0066,0.1251,0.0094
6,mol_MW,0.1168,0.0154,0.0431,0.3724
7,aqueous/organic,-0.1102,0.0222,-0.0641,0.1845
8,mol_melting_point,-0.0759,0.1159,-0.0356,0.4615
9,mol_Hacceptors,-0.0624,0.1965,-0.2244,0.0000


Saved: results\figures\Fig1_correlation_heatmap_EE.png | 300 DPI


## 4. Figure 2 — lipophilicity (`mol_logP`) vs EE%, colour-coded by PLGA LA/GA ratio
The LA/GA ratio **is** available in this dataset (5 discrete copolymer grades), so points are coloured by grade rather than by a continuous colour bar. A dashed OLS trend line is drawn for the pooled data as a visual reference only — **not** a fitted predictive model.

Two honest caveats are annotated on the figure: points are **not** independent (multiple formulations per drug and per study), and logP coverage is concentrated in the 0–5 band identified in the Phase-1 audit, so the extremes rest on very few drugs.

In [5]:
lv = np.sort(eda["LA_GA_ratio"].unique())
palette = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd", "#d62728", "#17becf", "#8c564b"]
cmap_d = {v: palette[i % len(palette)] for i, v in enumerate(lv)}

fig, ax = plt.subplots(figsize=(10.4, 6.4))
for v in lv:
    m = eda["LA_GA_ratio"] == v
    ax.scatter(eda.loc[m, "mol_logP"], eda.loc[m, TARGET],
               s=46, alpha=0.78, color=cmap_d[v], edgecolor="black", linewidth=0.45,
               label=f"{v:g}  (n={int(m.sum())})", zorder=3)

# pooled OLS reference line + 0-5 logP band shading
x, y = eda["mol_logP"].to_numpy(), eda[TARGET].to_numpy()
lr = stats.linregress(x, y)
xx = np.linspace(x.min(), x.max(), 100)
ax.plot(xx, lr.intercept + lr.slope * xx, "k--", lw=1.6, zorder=4,
        label=f"pooled OLS: r={lr.rvalue:.2f}, p={lr.pvalue:.1e}")
ax.axvspan(0, 5, color="grey", alpha=0.07, zorder=0)
ax.text(2.5, 106, "well-populated logP band (0–5)", ha="center", fontsize=8,
        color="#555555", style="italic")

# mark the retained EE = 0 formulation
z = eda.loc[eda["EE_is_zero"]]
if len(z):
    ax.scatter(z["mol_logP"], z[TARGET], s=210, facecolor="none", edgecolor="black",
               linewidth=1.8, zorder=6)
    ax.annotate("retained EE = 0\n(total encapsulation failure)",
                xy=(float(z["mol_logP"].iloc[0]), float(z[TARGET].iloc[0])),
                xytext=(float(z["mol_logP"].iloc[0]) + 1.15, 14),
                fontsize=8, ha="left",
                arrowprops=dict(arrowstyle="->", lw=1.0, color="black"))

ax.set_xlabel("mol_logP  (calculated lipophilicity, Crippen logP)")
ax.set_ylabel("Encapsulation efficiency, EE (%)")
ax.set_ylim(-5, 112)
ax.set_title("Figure 2. Drug lipophilicity vs encapsulation efficiency, by PLGA LA/GA ratio", fontsize=11.5)
leg = ax.legend(title="PLGA LA/GA ratio", fontsize=8.5, title_fontsize=9,
                loc="upper left", bbox_to_anchor=(1.015, 1.0), framealpha=0.95, ncols=1)
fig.text(0.5, -0.035,
         f"n = {len(eda)} formulations from {eda['reference'].nunique()} studies. Observations are NOT independent "
         "(repeated formulations per drug and per study); the dashed line is a visual reference, not a predictive model.\n"
         "logP coverage is concentrated in 0–5, so the extremes rest on very few distinct drugs — treat them as extrapolation.",
         ha="center", fontsize=8, style="italic", color="#444444")
out2 = FIG / "Fig2_logP_vs_EE_by_LAGA.png"
fig.savefig(out2, dpi=300); plt.close(fig)
print("Saved:", out2.relative_to(ROOT), "| 300 DPI")
print(f"\nPooled logP–EE: r = {lr.rvalue:.3f} (p = {lr.pvalue:.3g}), slope = {lr.slope:.2f} EE% per logP unit")
print("Per-LA/GA-grade logP–EE correlations (grades with n ≥ 10):")
for v in lv:
    m = eda["LA_GA_ratio"] == v
    if int(m.sum()) >= 10:
        rr, pp = stats.pearsonr(eda.loc[m, "mol_logP"], eda.loc[m, TARGET])
        print(f"   LA/GA {v:>5g}  n={int(m.sum()):>3d}   r={rr:+.3f}  p={pp:.3g}")

Saved: results\figures\Fig2_logP_vs_EE_by_LAGA.png | 300 DPI

Pooled logP–EE: r = 0.227 (p = 1.94e-06), slope = 3.77 EE% per logP unit
Per-LA/GA-grade logP–EE correlations (grades with n ≥ 10):
   LA/GA     1  n=349   r=+0.207  p=0.000101
   LA/GA     3  n= 76   r=+0.264  p=0.021


## 5. Figure 3 — distribution of EE%
Three views of the same variable: (a) histogram with a kernel-density curve, median/mean markers, and the retained EE = 0 point marked on a rug; (b) empirical cumulative distribution; (c) box + jittered strip plot. Together these show the left skew, the ceiling effect near 100%, and the sparse low-EE tail that will dominate prediction error in Phase 4.

In [6]:
fig = plt.figure(figsize=(15.4, 5.0))
gs  = fig.add_gridspec(1, 3, width_ratios=[1.5, 1.0, 0.72], wspace=0.46)
v = ee.to_numpy(float)

# ---- (a) histogram + KDE ----
ax1 = fig.add_subplot(gs[0, 0])
bins = np.arange(0, 105, 5)
ax1.hist(v, bins=bins, color="#5b8db8", edgecolor="black", linewidth=0.6, alpha=0.85)
ax1b = ax1.twinx(); ax1b.grid(False)
kde = stats.gaussian_kde(v)
xs = np.linspace(0, 100, 400)
ax1b.plot(xs, kde(xs), color="#b03a2e", lw=2.0)
ax1b.set_ylabel("kernel density", color="#b03a2e", fontsize=9)
ax1b.tick_params(axis="y", labelcolor="#b03a2e", labelsize=8)
ax1b.set_ylim(bottom=0)
ax1.axvline(ee.median(), color="black", ls="-",  lw=1.6, label=f"median = {ee.median():.1f}%")
ax1.axvline(ee.mean(),   color="black", ls="--", lw=1.6, label=f"mean = {ee.mean():.1f}%")
ax1.plot(v, np.full_like(v, -1.6), "|", color="#333333", ms=6, alpha=0.5, clip_on=False)
if len(z):
    ax1.plot([0], [-1.6], "|", color="#d62728", ms=13, mew=2.4, clip_on=False)
    ax1.annotate("EE = 0 (retained)", xy=(0, -1.6), xytext=(9, 15),
                 fontsize=8, color="#b03a2e",
                 arrowprops=dict(arrowstyle="->", color="#b03a2e", lw=1.0))
ax1.set_xlabel("Encapsulation efficiency, EE (%)"); ax1.set_ylabel("Number of formulations")
ax1.set_xlim(-2, 102)
ax1.set_title("(a) Histogram (5% bins) + kernel density", fontsize=10.5)
ax1.legend(fontsize=8, loc="upper left", framealpha=0.95)

# ---- (b) ECDF ----
ax2 = fig.add_subplot(gs[0, 1])
sv = np.sort(v); cdf = np.arange(1, len(sv) + 1) / len(sv)
ax2.step(sv, cdf, where="post", color="#1f4e79", lw=1.9)
for q, lab in [(.25, "Q1"), (.50, "median"), (.75, "Q3")]:
    xq = float(np.quantile(v, q))
    ax2.plot([xq, xq, -2], [0, q, q], color="grey", ls=":", lw=1.1)
    ax2.text(xq + 1.5, q - 0.06, f"{lab} = {xq:.1f}%", fontsize=8, color="#333333")
ax2.set_xlim(-2, 102); ax2.set_ylim(0, 1.02)
ax2.set_xlabel("EE (%)"); ax2.set_ylabel("Cumulative proportion")
ax2.set_title("(b) Empirical cumulative distribution", fontsize=10.5)

# ---- (c) box + strip ----
ax3 = fig.add_subplot(gs[0, 2])
bp = ax3.boxplot([v], widths=0.42, patch_artist=True, tick_labels=["all\nformulations"],
                 medianprops=dict(color="black", lw=1.8),
                 flierprops=dict(marker="", markersize=0))
bp["boxes"][0].set(facecolor="#aac6de", edgecolor="black", linewidth=0.8)
rng = np.random.default_rng(0)                    # fixed seed: jitter is cosmetic and reproducible
ax3.scatter(1 + rng.uniform(-0.13, 0.13, len(v)), v, s=13, alpha=0.45,
            color="#1f4e79", edgecolor="none", zorder=3)
ax3.set_ylabel("EE (%)"); ax3.set_ylim(-4, 104)
ax3.set_title("(c) Box + strip", fontsize=10.5)

fig.suptitle(f"Figure 3. Distribution of encapsulation efficiency (n = {len(eda)} formulations; "
             f"median {ee.median():.1f}%, IQR {ee.quantile(.25):.1f}–{ee.quantile(.75):.1f}%, "
             f"skew {stats.skew(v):.2f})", fontsize=12, y=1.04)
fig.text(0.5, -0.10,
         "EE% is bounded [0, 100], left-skewed, and non-normal (Shapiro p = "
         f"{stats.shapiro(v).pvalue:.2g}). The sparse low-EE tail — including one retained total-failure "
         "formulation — is where prediction error will concentrate; report MAE alongside RMSE in Phase 4.",
         ha="center", fontsize=8, style="italic", color="#444444")
out3 = FIG / "Fig3_EE_distribution.png"
fig.savefig(out3, dpi=300); plt.close(fig)
print("Saved:", out3.relative_to(ROOT), "| 300 DPI")

Saved: results\figures\Fig3_EE_distribution.png | 300 DPI


## 6. Verify the saved figures and log EDA observations

In [7]:
from PIL import Image as _PILImage  # bundled with matplotlib's dependencies
targets = ["Fig1_correlation_heatmap_EE.png", "Fig2_logP_vs_EE_by_LAGA.png", "Fig3_EE_distribution.png"]
rows = []
for f in targets:
    p = FIG / f
    assert p.exists(), f"MISSING figure: {f}"
    with _PILImage.open(p) as img:
        dpi = img.info.get("dpi", (None, None))
        rows.append({"figure": f, "exists": True, "KB": round(p.stat().st_size / 1024, 1),
                     "pixels": f"{img.size[0]}x{img.size[1]}", "dpi": f"{dpi[0]:.0f}"})
fig_check = pd.DataFrame(rows)
display(fig_check)
assert all(r["dpi"] == "300" for r in rows), "All figures must be 300 DPI."
print("All three required figures saved at 300 DPI.")

,figure,exists,KB,pixels,dpi
0,Fig1_correlation_heatmap_EE.png,True,601.9,4068x2368,300
1,Fig2_logP_vs_EE_by_LAGA.png,True,524.5,3552x1889,300
2,Fig3_EE_distribution.png,True,436.0,3925x1778,300


All three required figures saved at 300 DPI.


In [8]:
# --- EDA observation log (descriptive only; no modelling claims) ---
top3 = corr_ee.head(3)
obs = pd.DataFrame([
    ("target_distribution",
     f"EE median {ee.median():.1f}% (IQR {ee.quantile(.25):.1f}-{ee.quantile(.75):.1f}), "
     f"skew {stats.skew(v):.2f}, Shapiro p={stats.shapiro(v).pvalue:.2g}",
     "Left-skewed and bounded: report median/IQR and MAE alongside mean/SD and RMSE."),
    ("low_EE_tail",
     f"{int((ee<20).sum())} formulations below EE 20%, of which {int((ee==0).sum())} is a retained total failure",
     "Sparse tail; errors will concentrate here. Do not delete it - it is a real outcome."),
    ("strongest_linear_associations",
     "; ".join(f"{r.descriptor} r={r.pearson_r:+.2f} (p={r.pearson_p:.1g})" for r in top3.itertuples()),
     "All |r| are modest - no single descriptor determines EE. Multivariate, non-linear models are justified."),
    ("logP_vs_EE",
     f"pooled Pearson r={lr.rvalue:+.3f} (p={lr.pvalue:.3g}) across {len(eda)} formulations",
     "Weak pooled trend, confounded by study and polymer grade. Not a mechanistic lipophilicity claim."),
    ("laga_coverage",
     "; ".join(f"LA/GA {x:g}: n={int((eda['LA_GA_ratio']==x).sum())}" for x in lv),
     "Copolymer grades are unevenly sampled; grade-specific trends rest on unequal support."),
    ("non_independence",
     f"{len(eda)} formulations from {eda[GROUP_KEY].nunique()} drug groups and {eda['reference'].nunique()} studies",
     "Correlation p-values assume independence and are therefore optimistic. Use grouped CV in Phase 4."),
    ("pH_missing_category",
     f"pH_missing (undocumented -2 code) applies to {int(ml['pH_missing'].sum())} formulations",
     "Encoded as an explicit category, never imputed."),
], columns=["observation", "finding", "implication"])
obs.to_csv(TAB / "eda_observations.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(obs)

print("\n" + "=" * 62)
print("PHASE 3 (EDA) COMPLETE - no model trained")
print("-" * 62)
print(f"source          : data/processed/ML_ready_PLGA.csv  ({len(ml)} x {ml.shape[1]})")
print(f"figures (300dpi): {', '.join(targets)}")
print("tables          : eda_EE_summary.csv, eda_correlations_with_EE.csv,")
print("                  eda_correlation_matrix_pearson.csv, eda_observations.csv")
print("=" * 62)

,observation,finding,implication
0,target_distribution,"EE median 70.6% (IQR 52.4-83.2), skew -1.00, Shapiro p=1.6e-15",Left-skewed and bounded: report median/IQR and MAE alongside mean/SD and RMSE.
1,low_EE_tail,"32 formulations below EE 20%, of which 1 is a retained total failure",Sparse tail; errors will concentrate here. Do not delete it - it is a real outcome.
2,strongest_linear_associations,surfactant_concentration r=+0.31 (p=4e-11); surfactant_HLB r=+0.30 (p=2e-10); mol_logP r=+0.23 (p=2e-06),"All |r| are modest - no single descriptor determines EE. Multivariate, non-linear models are justified."
3,logP_vs_EE,pooled Pearson r=+0.227 (p=1.94e-06) across 430 formulations,"Weak pooled trend, confounded by study and polymer grade. Not a mechanistic lipophilicity claim."
4,laga_coverage,LA/GA 1: n=349; LA/GA 1.86: n=3; LA/GA 2.33: n=1; LA/GA 3: n=76; LA/GA 5.67: n=1,Copolymer grades are unevenly sampled; grade-specific trends rest on unequal support.
5,non_independence,430 formulations from 63 drug groups and 59 studies,Correlation p-values assume independence and are therefore optimistic. Use grouped CV in Phase 4.
6,pH_missing_category,pH_missing (undocumented -2 code) applies to 26 formulations,"Encoded as an explicit category, never imputed."



PHASE 3 (EDA) COMPLETE - no model trained
--------------------------------------------------------------
source          : data/processed/ML_ready_PLGA.csv  (430 x 31)
figures (300dpi): Fig1_correlation_heatmap_EE.png, Fig2_logP_vs_EE_by_LAGA.png, Fig3_EE_distribution.png
tables          : eda_EE_summary.csv, eda_correlations_with_EE.csv,
                  eda_correlation_matrix_pearson.csv, eda_observations.csv


## Phase 3 complete

`data/processed/ML_ready_PLGA.csv` is on disk and all three required figures are saved to `results/figures/` at 300 DPI.

**Headline observations (descriptive only):**
- EE% is **left-skewed and bounded** — most formulations achieve high encapsulation, with a thin, sparsely-populated low-EE tail that includes one retained total-failure (EE = 0) formulation.
- **No single descriptor determines EE.** All descriptor–EE correlations are modest in magnitude, which is what justifies a multivariate, non-linear model (e.g. SVR) rather than a simple regression on lipophilicity.
- The `logP`–EE relationship is **weak when pooled** and is confounded with study identity and polymer grade; `LA/GA` grades are unevenly sampled.
- Reported p-values assume independent observations, which these are **not** (repeated formulations per drug and per study) — so they are optimistic. This is precisely why Phase 4 must evaluate with **`GroupKFold` on `drug_group`**.

**Stopping condition honoured: no machine-learning model has been trained.** Phase 4 (modelling) is the next step and has not been started.